<a href="https://colab.research.google.com/github/Aymanyah/Stat_app/blob/main/notebook/scouting_statapp_Raph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [88]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
import re
import unicodedata
from sklearn.preprocessing import StandardScaler

In [ ]:
import pandas as pd

#url = "https://raw.githubusercontent.com/Aymanyah/Stat_app/main/data/clean/dataset_aggregated.csv"
#f = pd.read_csv(url)

#df.head()

,player,season,age,born,Minutes de jeu,# 90 min jouées,G,A,G+A,G-PK,G+A-PK,xG,xAG,xG+xAG,npxG,npxG+xAG,% dribble réussis,% passes réussies,% passes courtes réussies,% passes moyennes réussies,% passes longues réussies,% duels aériens gagnés,xG / Tir,Valeur marchande (euros),Chevauchée avant >5m,Passes avant >10m,Passes > 10m vers l'avant reçues,Tacles réussis qui gagnent la balle,Dribbles subis perdus,Blocs totaux,Interceptions,Tacles réussis + interceptions,Dégagements,Erreurs menant à un tir adverse,Tirs,Tirs cadrés,Distance totale des passes > 10m avant effectuées,Passes menant à un but,Touches ballon en jeu,Passes reçues,team,league,nation,pos
0,Aaron Connolly,2021.0,20.0,2000.0,791.0,8.8,0.23,0.11,0.34,0.23,0.34,0.45,0.02,0.48,0.45,0.48,0.454545,0.766,0.785,0.875,0.000000,0.184211,0.173913,6000000.0,1.363636,0.454545,6.590909,0.681818,0.227273,0.454545,0.000000,0.681818,0.227273,0.000000,2.613636,0.909091,20.000000,0.681818,19.772727,15.227273,Brighton,ENG-Premier League,IRL,FW
1,Aaron Cresswell,2020.0,30.0,1989.0,3170.0,35.2,0.00,0.23,0.23,0.00,0.23,0.03,0.18,0.21,0.03,0.21,0.400000,0.741,0.890,0.803,0.461538,0.546875,0.052632,5000000.0,0.994318,4.602273,2.727273,0.909091,0.397727,0.823864,0.909091,1.818182,2.244318,0.056818,0.539773,0.113636,393.323864,1.647727,67.215909,38.323864,West Ham,ENG-Premier League,ENG,DF
2,Aaron Cresswell,2021.0,30.0,1989.0,3170.0,35.2,0.00,0.23,0.23,0.00,0.23,0.03,0.18,0.21,0.03,0.21,0.400000,0.741,0.890,0.803,0.461538,0.546875,0.052632,3000000.0,0.994318,4.602273,2.727273,0.909091,0.397727,0.823864,0.909091,1.818182,2.244318,0.056818,0.539773,0.113636,393.323864,1.647727,67.215909,38.323864,West Ham,ENG-Premier League,ENG,DF
3,Aaron Cresswell,2022.0,32.0,1989.0,2235.0,24.8,0.00,0.04,0.04,0.00,0.04,0.01,0.15,0.16,0.01,0.16,0.285714,0.778,0.934,0.779,0.485437,0.509091,0.033333,1200000.0,1.411290,5.846774,3.306452,0.846774,0.766129,0.846774,1.088710,1.935484,2.258065,0.000000,0.362903,0.040323,346.250000,1.612903,64.798387,38.709677,West Ham,ENG-Premier League,ENG,DF
4,Aaron Hickey,2020.0,18.0,2002.0,758.0,8.4,0.00,0.00,0.00,0.00,0.00,0.01,0.01,0.01,0.01,0.01,0.437500,0.849,0.920,0.861,0.617647,0.545455,0.050000,5000000.0,2.500000,4.285714,3.333333,1.428571,1.190476,0.714286,1.785714,3.214286,1.785714,0.119048,0.238095,0.000000,207.380952,0.238095,58.928571,31.904762,Bologna,ITA-Serie A,SCO,DF


In [45]:
# --- 1) RECONVERTIR LES CHAÎNES EN UTF-8 CORRECT ---
# Nettoyer les noms de colonnes
df = pd.read_csv("all_leagues_merged_transformed.csv", encoding="utf8")

def fix_text(s):
    if isinstance(s, str):
        try:
            # On réencode et on décode pour corriger les caractères bizarres
            return s.encode('latin1').decode('utf-8')
        except:
            return s
    return s

# Appliquer à toutes les colonnes texte
text_cols = ["player", "team", "league", "nation", "pos"]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).apply(fix_text)

# --- 2) Nettoyage simple des colonnes ---
df.columns = [fix_text(col).strip() for col in df.columns]

# Supprimer les doublons exacts
df_clean = df.drop_duplicates()

# --- 1) COLONNES NUMÉRIQUES ---
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ["season"]]

# --- 2) DÉFINITION D'UNE FONCTION D'AGRÉGATION INTELLIGENTE ---
def smart_agg(col):
    if "percent" in col.lower() or "%" in col or "ratio" in col.lower():
        return "mean"
    if col in ["Minutes de jeu", "G", "G-PK", "Assists", "Tirs", "Tirs cadrés"]:
        return "sum"
    return "mean"

agg_dict = {col: smart_agg(col) for col in numeric_cols}

# --- 3) AJOUT DES COLONNES CATEGORIELLES ---
# On prend "last" pour team, league, nation, pos
categorical_cols = [c for c in ["team", "league", "nation", "pos"] if c in df_clean.columns]
for col in categorical_cols:
    agg_dict[col] = "last"

# --- 4) AGGRÉGATION PAR JOUEUR + SAISON ---
df_grouped = df_clean.groupby(["player", "season"]).agg(agg_dict).reset_index()

# --- 5) PIVOT DES STATISTIQUES NUMÉRIQUES ---
df_stats = df_grouped.pivot(index="player", columns="season", values=numeric_cols)
df_stats.columns = [f"{stat}_{season}" for stat, season in df_stats.columns]
df_stats = df_stats.reset_index()

# --- 6) PIVOT DES COLONNES CATEGORIELLES ---
df_cats = {}
for col in categorical_cols:
    df_cat = df_grouped.pivot(index="player", columns="season", values=col)
    df_cat.columns = [f"{col}_{season}" for season in df_cat.columns]
    df_cats[col] = df_cat.reset_index()

# --- 7) FUSION DE TOUTES LES COLONNES ---
df_final = df_stats
for col, df_cat in df_cats.items():
    df_final = df_final.merge(df_cat, on="player", how="left")

# --- 8) SUPPRESSION INTELLIGENTE DES COLONNES 2021 POUR LES STATS NUMÉRIQUES ---
for col in df_final.columns:
    if col.endswith("_2020.0"):
        col_2021 = col.replace("_2020.0", "_2021.0")
        if col_2021 in df_final.columns:
            df_final[col] = df_final[col].fillna(df_final[col_2021])
            df_final = df_final.drop(columns=[col_2021])

# --- 9) GARDER UNE SEULE COLONNE BORN ET POS ---
born_cols = [c for c in df_final.columns if c.startswith("born_")]
if born_cols:
    df_final["born"] = df_final[born_cols].bfill(axis=1).iloc[:, 0]
    df_final = df_final.drop(columns=born_cols)

pos_cols = [c for c in df_final.columns if c.startswith("pos_")]
if pos_cols:
    df_final["pos"] = df_final[pos_cols].bfill(axis=1).iloc[:, 0]
    df_final = df_final.drop(columns=pos_cols)


# --- 10) METTRE EN % LES VALEURS EN POURCENTAGE ---
percent_cols = [c for c in df_final.columns if "%" in c]
for col in percent_cols:
    if df_final[col].mean(skipna=True) > 1:
        df_final[col] = df_final[col] / 100


print(df_final.head())

# Exemple : récupérer toutes les informations de Harry Kane
#player_name = "Neal Maupay"
pd.set_option('display.max_columns', None)  # Montre toutes les colonnes
pd.set_option('display.width', 3000)
df_joueur = df_final[df_final["player"] == player_name]

# Afficher le résultat
#print(df_joueur)


             player  age_2020  age_2021  age_2022  age_2023  Minutes de jeu_2020  Minutes de jeu_2021  Minutes de jeu_2022  Minutes de jeu_2023  # 90 min jouées_2020  # 90 min jouées_2021  # 90 min jouées_2022  # 90 min jouées_2023  G_2020  G_2021  G_2022  G_2023  A_2020  A_2021  A_2022  A_2023  G+A_2020  G+A_2021  G+A_2022  G+A_2023  G-PK_2020  G-PK_2021  G-PK_2022  G-PK_2023  G+A-PK_2020  G+A-PK_2021  G+A-PK_2022  G+A-PK_2023  xG_2020  xG_2021  xG_2022  xG_2023  xAG_2020  xAG_2021  xAG_2022  xAG_2023  xG+xAG_2020  xG+xAG_2021  xG+xAG_2022  xG+xAG_2023  npxG_2020  npxG_2021  npxG_2022  npxG_2023  npxG+xAG_2020  npxG+xAG_2021  npxG+xAG_2022  npxG+xAG_2023  % dribble  réussis_2020  % dribble  réussis_2021  % dribble  réussis_2022  % dribble  réussis_2023  % passes réussies_2020  % passes réussies_2021  % passes réussies_2022  % passes réussies_2023  % passes courtes réussies_2020  % passes courtes réussies_2021  % passes courtes réussies_2022  % passes courtes réussies_2023  % passes mo

In [49]:
numeric_cols = df_final.select_dtypes(include=[np.number]).columns

# Retirer le suffixe saison (_2020, _2022, etc.) pour ne garder que le nom de la stat
stat_names = set(col.split("_")[0] for col in numeric_cols)

# Afficher toutes les stats uniques
print(sorted(stat_names))

['# 90 min jouées', '% dribble  réussis', '% duels aériens gagnés', '% passes courtes réussies', '% passes longues réussies', '% passes moyennes réussies', '% passes réussies', 'A', 'Blocs totaux', 'Chevauchée avant  >5m', 'Distance totale des passes > 10m avant effectuées', 'Dribbles subis perdus', 'Dégagements', 'Erreurs menant à un tir adverse', 'G', 'G+A', 'G+A-PK', 'G-PK', 'Interceptions', 'Minutes de jeu', "Passes > 10m vers l'avant reçues", 'Passes avant >10m', 'Passes menant à un but', 'Passes reçues', 'Tacles réussis + interceptions', 'Tacles réussis qui gagnent la balle', 'Tirs', 'Tirs cadrés', 'Touches ballon en jeu', 'Valeur marchande (euros)', 'age', 'born', 'npxG', 'npxG+xAG', 'xAG', 'xG', 'xG / Tir', 'xG+xAG']


In [114]:
# Trouver 20 meilleurs attaquants et millieux par rapp à l'offensive Inde

# Joueurs ayant joué au moins 10 matchs (~900 min)
df_last2 = df_final[df_final["Minutes de jeu_2023"] >= 900.0].copy()

# Coefficients ligues
coeff_dict = {
    "ENG-Premier League": 10.4303,
    "ITA-Serie A": 9.0284,
    "ESP-La Liga": 8.9489,
    "GER-Bundesliga": 8.6624,
    "FRA-Ligue 1": 6.6831,
}
df_last2["uefa_coefficient_2023"] = df_last2["league_2023"].map(coeff_dict)

# Réduire le poids du % de passes réussies
df_last2["passes réussies ratio"] = df_last2["% passes réussies_2023"] / 100

# ===============================
# 1️⃣ Offensive Strength Index
# ===============================
df_last2["OSI"] = (
    df_last2["G-PK_2023"] * 4
    + df_last2["G-PK_2023"] * 2
    + (df_last2["G_2023"] - df_last2["G-PK_2023"]) * 0.5
    + df_last2["Tirs_2023"] * 0.5
    + df_last2["Tirs cadrés_2023"] * 1.8
    + df_last2["xG_2023"] * 1.2
    + df_last2["xG / Tir_2023"] * 0.5
    + df_last2["% duels aériens gagnés_2023"] * 0.01
    + df_last2["Passes menant à un but_2023"] * 0.5
    + df_last2["uefa_coefficient_2023"]
)
df_last2["Off Index"] = (
    (df_last2["OSI"] - df_last2["OSI"].min())
    / (df_last2["OSI"].max() - df_last2["OSI"].min())
) * 100

def plot_top20_interactive_off(df_subset, title="Top 20 Players – Off Index", stats_cols=None, cmap_name="Reds"):
    """
    df_subset : dataframe des 20 meilleurs attaquants/midfielders
    stats_cols : liste des colonnes utilisées pour calculer l'indice Off Index
    """
    players = df_subset["player"]
    values = df_subset["Off Index"]

    hover_texts = []
    for i, row in df_subset.iterrows():
        hover_text = f"<b>Off Index : {row['Off Index']:.1f}%</b><br>"  # Indice en haut
        hover_text += f"<b>{row['player']}</b><br>"
        hover_text += f"Poste : {row['pos']}<br>"
        hover_text += f"Club : {row['team_2023']}<br>"
        hover_text += f"Ligue : {row['league_2023']}<br>"
        hover_text += f"Age : {row['age_2023']}<br><br>"
        hover_text += "<b>Stats utilisées :</b><br>"
        for col in stats_cols:
            if col in row:
                hover_text += f"{col}: {row[col]}<br>"
        hover_texts.append(hover_text)

    fig = go.Figure(go.Bar(
        x=values,
        y=players,
        orientation='h',
        text=[f"{v:.1f}%" for v in values],
        textposition='outside',
        marker=dict(
            color=values,
            colorscale=cmap_name,
            showscale=True
        ),
        hoverinfo='text',
        hovertext=hover_texts
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Offensive Strength Index (%)",
        yaxis=dict(autorange="reversed"),  # meilleur en haut
        height=600,
        margin=dict(l=150, r=50, t=50, b=50)
    )

    fig.show()

stats_cols_off = [
    "G-PK", "G", "Tirs", "Tirs cadrés", "xG", "xG / Tir", "% duels aériens gagnés", "uefa_coefficient"
]

# Plot interactif pour attaquants
plot_top20_interactive_off(df_attack, title="Top 20 Attackers – Off Index", stats_cols=stats_cols_off, cmap_name="Reds")

# Plot interactif pour milieux
plot_top20_interactive_off(df_attack, title="Top 20 Midfielders – Off Index", stats_cols=stats_cols_off, cmap_name="Purples")



In [79]:
# Trouver 20 meilleurs  millieux par rapp Creativity Index

# Joueur avec min 10 matchs joués sur la saison
df_last2 = df_final[df_final["Minutes de jeu_2023"] >= 900.0].copy()

# Réduire le poids du % de passes réussies
df_last2["passes réussies ratio_2023"] = df_last2["% passes réussies_2023"] / 100

# Ajouter la colonne coefficient dans ton df
coeff_dict = {
    "ENG-Premier League": 10.4303,
    "ITA-Serie A": 9.0284,
    "ESP-La Liga": 8.9489,
    "GER-Bundesliga": 8.6624,
    "FRA-Ligue 1": 6.6831,
}
df_last2["uefa_coefficient_2023"] = df_last2["league_2023"].map(coeff_dict)

# Calcul de l'indice Mil Index
df_last2["MSI"] = (
    # Création / offensif
    df_last2["A_2023"] * 2
    + df_last2["G_2023"] * 0.5
    + df_last2["passes réussies ratio_2023"] * 3
    + df_last2["Passes menant à un but_2023"] * 1
    + df_last2["Passes avant >10m_2023"] * 0.5
    + df_last2["% passes longues réussies_2023"] * 0.6
    + df_last2["xAG_2023"] * 1
    + df_last2["% dribble  réussis_2023"] * 0.5

    # Défensif
    + df_last2["Interceptions_2023"] * 1.8
    + df_last2["Tacles réussis qui gagnent la balle_2023"] * 1.5
    + df_last2["Dégagements_2023"] * 1
    + df_last2["Blocs totaux_2023"] * 1.5

    # Participation / volume
    + df_last2["Minutes de jeu_2023"] * 0.05
    + df_last2["Touches ballon en jeu_2023"] * 1
    + df_last2["Chevauchée avant  >5m_2023"] * 0.5
    + df_last2["Distance totale des passes > 10m avant effectuées_2023"] * 0.5

    # Poids ligue
    + df_last2["uefa_coefficient_2023"]
)

# Normalisation 0–100%
df_last2["Mil Index"] = (
    (df_last2["MSI"] - df_last2["MSI"].min())
    / (df_last2["MSI"].max() - df_last2["MSI"].min())
) * 100

# Tri
df_sorted = df_last2.sort_values("Mil Index", ascending=False)

# Sélection des 20 meilleurs milieux
df_mid = df_sorted[df_sorted["pos"] == "MF"].head(20)


# ----- Plot interactif -----
import plotly.graph_objects as go

def plot_top20_interactive(df_subset, title="Top 20 Midfielders – Mil Index", stats_cols=None):
    players = df_subset["player"]
    values = df_subset["Mil Index"]

    hover_texts = []
    for i, row in df_subset.iterrows():
        hover_text = f"<b>Mil Index : {row['Mil Index']:.1f}%</b><br>"
        hover_text += f"<b>{row['player']}</b><br>"
        hover_text += f"Poste : {row['pos']}<br>"
        hover_text += f"Club : {row['team_2023']}<br>"
        hover_text += f"Ligue : {row['league_2023']}<br>"
        hover_text += f"Age : {row['age_2023']}<br><br>"
        hover_text += "<b>Stats utilisées :</b><br>"
        for col in stats_cols:
            if col in row:
                clean_name = col.replace("_2023", "")
                hover_text += f"{clean_name}: {row[col]}<br>"
        hover_texts.append(hover_text)

    fig = go.Figure(go.Bar(
        x=values,
        y=players,
        orientation='h',
        text=[f"{v:.1f}%" for v in values],
        textposition='outside',
        marker=dict(
            color=values,
            colorscale='Oranges',
            showscale=True
        ),
        hoverinfo='text',
        hovertext=hover_texts
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Mil Index (%)",
        yaxis=dict(autorange="reversed"),
        height=600,
        margin=dict(l=150, r=50, t=50, b=50)
    )

    fig.show()


# Colonnes utilisées pour MSI
stats_cols_2023 = [
    "A_2023", "G_2023", "passes réussies ratio_2023", "Passes menant à un but_2023", "Passes avant >10m_2023",
    "% passes longues réussies_2023", "xAG_2023", "% dribble  réussis_2023",
    "Interceptions_2023", "Tacles réussis qui gagnent la balle_2023", "Dégagements_2023", "Blocs totaux_2023",
    "Minutes de jeu_2023", "Touches ballon en jeu_2023", "Chevauchée avant  >5m_2023", "Distance totale des passes > 10m avant effectuées_2023"
]

# Plot interactif
plot_top20_interactive(df_mid, stats_cols=stats_cols_2023)

/tmp/ipython-input-1030/3733312185.py:9: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipython-input-1030/3733312185.py:21: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipython-input-1030/3733312185.py:28: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/

In [108]:
# Joueur avec min 10 matchs joués sur la saison
df_last2 = df_final[df_final["Minutes de jeu_2023"] >= 900].copy()

# Réduire le poids du % de passes réussies
df_last2["passes réussies ratio_2023"] = df_last2["% passes réussies_2023"] / 100

# Ajouter la colonne coefficient
coeff_dict = {
    "ENG-Premier League": 10.4303,
    "ITA-Serie A": 9.0284,
    "ESP-La Liga": 8.9489,
    "GER-Bundesliga": 8.6624,
    "FRA-Ligue 1": 6.6831,
}
df_last2["uefa_coefficient_2023"] = df_last2["league_2023"].map(coeff_dict)

# Construire l’indice défensif
df_last2["DSI"] = (
    df_last2["Tacles réussis qui gagnent la balle_2023"] * 2
    + df_last2["Tacles réussis + interceptions_2023"] * 3
    + df_last2["Interceptions_2023"] * 2
    + df_last2["Blocs totaux_2023"] * 2
    + df_last2["Dégagements_2023"] * 0.01
    + df_last2["% duels aériens gagnés_2023"] * 2
    + df_last2["passes réussies ratio_2023"] * 1
    + df_last2["% passes longues réussies_2023"] * 1
    + df_last2["uefa_coefficient_2023"]
    + df_last2["Minutes de jeu_2023"] * 0.1
    - df_last2["Dribbles subis perdus_2023"] * 1.2
    - df_last2["Erreurs menant à un tir adverse_2023"] * 4
)

# Normalisation 0–100%
df_last2["Def Index"] = (
    (df_last2["DSI"] - df_last2["DSI"].min())
    / (df_last2["DSI"].max() - df_last2["DSI"].min())
) * 100

# Tri
df_sorted = df_last2.sort_values("Def Index", ascending=False)

# Sélection défenseurs et milieux
df_def = df_sorted[df_sorted["pos"] == "DF"].head(20)
df_mid = df_sorted[df_sorted["pos"] == "MF"].head(20)


# ----- Plot interactif -----
import plotly.graph_objects as go

def plot_top20_interactive_def(df_subset, title="Top 20 Defenders – Def Index", stats_cols=None, cmap_name="Greens"):
    players = df_subset["player"]
    values = df_subset["Def Index"]

    hover_texts = []
    for i, row in df_subset.iterrows():
        hover_text = f"<b>Def Index : {row['Def Index']:.1f}%</b><br>"
        hover_text += f"<b>{row['player']}</b><br>"
        hover_text += f"Poste : {row['pos']}<br>"
        hover_text += f"Club : {row['team_2023']}<br>"
        hover_text += f"Ligue : {row['league_2023']}<br>"
        hover_text += f"Age : {row['age_2023']}<br><br>"
        hover_text += "<b>Stats utilisées :</b><br>"
        for col in stats_cols:
            if col in row:
                clean_name = col.replace("_2023", "")
                hover_text += f"{clean_name}: {row[col]}<br>"
        hover_texts.append(hover_text)

    fig = go.Figure(go.Bar(
        x=values,
        y=players,
        orientation='h',
        text=[f"{v:.1f}%" for v in values],
        textposition='outside',
        marker=dict(
            color=values,
            colorscale=cmap_name,
            showscale=True
        ),
        hoverinfo='text',
        hovertext=hover_texts
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Defensive Strength Index (%)",
        yaxis=dict(autorange="reversed"),
        height=600,
        margin=dict(l=150, r=50, t=50, b=50)
    )

    fig.show()


# Colonnes utilisées pour Def Index
stats_cols_def_2023 = [
    "Tacles réussis qui gagnent la balle_2023",
    "Tacles réussis + interceptions_2023",
    "Interceptions_2023",
    "Blocs totaux_2023",
    "Dégagements_2023",
    "% duels aériens gagnés_2023",
    "passes réussies ratio_2023",
    "% passes longues réussies_2023",
    "Dribbles subis perdus_2023",
    "Erreurs menant à un tir adverse_2023",
    "uefa_coefficient_2023",
    "Minutes de jeu_2023"
]

# Plot interactif défenseurs
plot_top20_interactive_def(df_def, title="Top 20 Defenders – Def Index", stats_cols=stats_cols_def_2023, cmap_name="Blues")

# Plot interactif milieux
plot_top20_interactive_def(df_mid, title="Top 20 Midfielders – Def Index", stats_cols=stats_cols_def_2023, cmap_name="Greens")

In [87]:
#Créations de pleins d'autres indices

df_last2 = df_clean[df_clean["season"] == 2023].copy()

#joueur avec min 10 matchs joués sur la saison
df_last2 = df_last[df_last["Minutes de jeu"] >= 900.0]

#réduire le poids du % de passes réussies
df_last2["passes réussies ratio"] = df_last2["% passes réussies"] / 100

#Ajouter une ligne au data set df_last
coeff_dict = {
    "ENG-Premier League": 10.4303,
    "ITA-Serie A": 9.0284,
    "ESP-La Liga": 8.9489,
    "GER-Bundesliga": 8.6624,
    "FRA-Ligue 1": 6.6831,
}

# Ajouter la colonne coefficient dans ton df
df_last2["uefa_coefficient"] = df_last2["league"].map(coeff_dict)


# Vérifie
df_last2.head()

#Indice d'efficacité offensive du joueur

from sklearn.preprocessing import StandardScaler

cols = [
    "G",
    "A",
    "Tirs cadrés",
    "shot_accuracy",
    "Passes menant à un but",
    "% passes réussies",
    "uefa_coefficient"
]

scaler = StandardScaler()
pei_scaled = pd.DataFrame(
    scaler.fit_transform(df[pei_cols]),
    columns=pei_cols
)

df["PEIndex"] = (
    pei_scaled["G"] * 2
    + pei_scaled["A"] * 1.5
    + pei_scaled["Tirs cadrés"] * 1.5
    + pei_scaled["shot_accuracy"] * 1.5
    + pei_scaled["Passes menant à un but"] * 0.8
    + pei_scaled["% passes réussies"] * 1.2
    + pei_scaled["uefa_coefficient"]
)

# Normalisation 0–100%
df_last2["Player_Efficiency_Index"] = (
    (df_last2["PEIndex"] - df_last2["PEIndex"].min())
    / (df_last2["PEIndex"].max() - df_last2["PEIndex"].min())
) * 100

/tmp/ipython-input-1030/3539552725.py:9: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipython-input-1030/3539552725.py:21: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



NameError: name 'StandardScaler' is not defined